In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# --- Load data ---
df = pd.read_csv("mesocases.csv")

# --- Anonymise ---
df = df.drop(columns=["MR NO", "Pt Name"], errors="ignore")

# --- Clean variables ---
df["date"] = pd.to_datetime(df.get("date"), errors="coerce")

df["Gender"] = (
    df.get("Gender")
    .astype(str)
    .str.strip()
    .str.title()
    .replace({"Nan": np.nan, "": np.nan})
)

df["smoking"] = (
    df.get("smoking")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "nan": np.nan,
        "": np.nan,
        "non smoker": np.nan,
        "cigarrete smoker": "Current smoker",
        "current smoker": "Current smoker",
        "ex smoker": "Ex-smoker",
        "ex-cigarette smoker": "Ex-smoker",
        "ex-cigarrete smoker": "Ex-smoker",
    })
)

# --- Biopsy diagnosis: 2 categories ---
biopsy_raw = (
    df.get("biopsy report")
    .astype(str)
    .str.strip()
    .str.lower()
)

df["biopsy_2cat"] = np.where(
    biopsy_raw.str.contains("epith", na=False),
    "Epithelioid mesothelioma",
    "Mesothelioma, subtype not specified"
)

df.loc[biopsy_raw.isin(["", "nan", "none"]), "biopsy_2cat"] = np.nan

biopsy_order = [
    "Epithelioid mesothelioma",
    "Mesothelioma, subtype not specified"
]

# --- Address aggregation ---
df["address_clean"] = (
    df.get("Address")
    .astype(str)
    .str.strip()
    .str.title()
    .replace({
        "Nan": np.nan,
        "": np.nan,
        "Afganistan": "Afghanistan",
        "Bajaur Agency": "Bajaur",
        "Mohmand Agency": "Mohmand",
    })
)

area_counts = df["address_clean"].value_counts(dropna=False)

df["area_grouped"] = df["address_clean"].where(
    df["address_clean"].map(area_counts) >= 3,
    "Other / small numbers"
)

# --- Helper functions ---
def n_pct(series, order=None, other_last=True):
    total = len(series)
    counts = series.value_counts(dropna=False)

    if order is not None:
        ordered_counts = {}
        for item in order:
            if item == "Missing":
                ordered_counts[np.nan] = counts.get(np.nan, 0)
            else:
                ordered_counts[item] = counts.get(item, 0)
        counts = pd.Series(ordered_counts)

    elif other_last and "Other / small numbers" in counts.index:
        other_count = counts.loc["Other / small numbers"]
        counts = counts.drop(index="Other / small numbers")
        counts = pd.concat([counts, pd.Series({"Other / small numbers": other_count})])

    out = []
    for k, v in counts.items():
        label = "Missing" if pd.isna(k) else k
        out.append([label, f"{int(v)} ({v / total * 100:.1f}%)"])
    return out

def median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1, med, q3 = s.quantile([0.25, 0.5, 0.75])
    return f"{med:.1f} ({q1:.1f}–{q3:.1f})"

def age_range(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    return f"{s.min():.0f}–{s.max():.0f}"

def style_table(table, title):
    category_rows = table.index[table.iloc[:, 1:].eq("").all(axis=1)].tolist()

    def style_rows(row):
        if row.name in category_rows:
            return [
                "font-weight: bold; background-color: #f2f2f2; text-align: left;"
                for _ in row
            ]
        return ["text-align: left;" for _ in row]

    return (
        table.style
        .hide(axis="index")
        .set_caption(title)
        .apply(style_rows, axis=1)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-weight", "bold"),
                    ("font-size", "14pt"),
                    ("text-align", "left"),
                    ("margin-bottom", "10px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("font-weight", "bold"),
                    ("border-bottom", "2px solid black"),
                    ("text-align", "left"),
                    ("padding", "6px 10px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "6px 10px"),
                    ("border-bottom", "1px solid #ddd"),
                    ("text-align", "left"),
                ],
            },
        ])
    )

# ============================================================
# Table 1: Overall cohort
# ============================================================

n = len(df)
rows = []

rows.append(["Age", ""])
rows.append(["  Median (IQR)", median_iqr(df["Age"])])
rows.append(["  Range", age_range(df["Age"])])

rows.append(["Sex", ""])
for label, value in n_pct(df["Gender"]):
    rows.append([f"  {label}", value])

rows.append(["Smoking status", ""])
smoking_order = ["Current smoker", "Ex-smoker", "Missing"]
for label, value in n_pct(df["smoking"], order=smoking_order):
    rows.append([f"  {label}", value])

rows.append(["Biopsy diagnosis", ""])
for label, value in n_pct(df["biopsy_2cat"], order=biopsy_order):
    rows.append([f"  {label}", value])

rows.append(["Area of residence", ""])
for label, value in n_pct(df["area_grouped"], other_last=True):
    rows.append([f"  {label}", value])

table1 = pd.DataFrame(rows, columns=["Characteristic", f"Overall cohort (N={n})"])

display(style_table(table1, "Table 1. Baseline characteristics of mesothelioma cases"))

display(HTML("""
<div style="font-size:10pt; margin-top:6px; margin-bottom:20px">
Values are n (%) unless otherwise stated. Residential address was aggregated to area/district level.
Areas with fewer than 3 cases were combined as “Other / small numbers” to reduce disclosure risk.
IQR = interquartile range.
</div>
"""))

# ============================================================
# Table 2: Stratified by sex
# ============================================================

sex_order = [x for x in ["Male", "Female"] if x in df["Gender"].dropna().unique()]
sex_order += [x for x in df["Gender"].dropna().unique() if x not in sex_order]

sex_cols = [f"{sex} (N={(df['Gender'] == sex).sum()})" for sex in sex_order]

def strat_row(label, variable, order=None, other_last=True):
    row = [label]
    for sex in sex_order:
        d = df[df["Gender"] == sex]
        values = dict(n_pct(d[variable], order=order, other_last=other_last))
        clean_label = label.strip()
        row.append(values.get(clean_label, "0 (0.0%)"))
    return row

rows2 = []

rows2.append(["Age", *["" for _ in sex_order]])
rows2.append(["  Median (IQR)", *[median_iqr(df.loc[df["Gender"] == sex, "Age"]) for sex in sex_order]])
rows2.append(["  Range", *[age_range(df.loc[df["Gender"] == sex, "Age"]) for sex in sex_order]])

rows2.append(["Smoking status", *["" for _ in sex_order]])
for label in smoking_order:
    rows2.append(strat_row(f"  {label}", "smoking", order=smoking_order))

rows2.append(["Biopsy diagnosis", *["" for _ in sex_order]])
for label in biopsy_order:
    rows2.append(strat_row(f"  {label}", "biopsy_2cat", order=biopsy_order))

rows2.append(["Area of residence", *["" for _ in sex_order]])
area_order = list(df["area_grouped"].value_counts().index)
if "Other / small numbers" in area_order:
    area_order = [x for x in area_order if x != "Other / small numbers"] + ["Other / small numbers"]

for label in area_order:
    rows2.append(strat_row(f"  {label}", "area_grouped", order=area_order))

table2 = pd.DataFrame(rows2, columns=["Characteristic", *sex_cols])

display(style_table(table2, "Table 2. Baseline characteristics of mesothelioma cases stratified by sex"))

display(HTML("""
<div style="font-size:10pt; margin-top:6px">
Values are n (%) within sex column unless otherwise stated. Residential address was aggregated to area/district level.
Areas with fewer than 3 cases were combined as “Other / small numbers” to reduce disclosure risk.
IQR = interquartile range.
</div>
"""))

# --- Export ---
table1.to_csv("table1_mesothelioma_cases.csv", index=False)
table2.to_csv("table2_mesothelioma_cases_by_sex.csv", index=False)

table1.to_excel("table1_mesothelioma_cases.xlsx", index=False)
table2.to_excel("table2_mesothelioma_cases_by_sex.xlsx", index=False)

Characteristic,Overall cohort (N=38)
Age,
Median (IQR),60.0 (50.0–68.8)
Range,29–100
Sex,
Male,24 (63.2%)
Female,14 (36.8%)
Smoking status,
Current smoker,2 (5.3%)
Ex-smoker,3 (7.9%)
Missing,33 (86.8%)


Characteristic,Male (N=24),Female (N=14)
Age,,
Median (IQR),59.5 (50.0–67.0),65.0 (52.5–68.8)
Range,29–83,36–100
Smoking status,,
Current smoker,2 (8.3%),0 (0.0%)
Ex-smoker,3 (12.5%),0 (0.0%)
Missing,19 (79.2%),14 (100.0%)
Biopsy diagnosis,,
Epithelioid mesothelioma,13 (54.2%),6 (42.9%)
"Mesothelioma, subtype not specified",11 (45.8%),8 (57.1%)
